Weights and Biases Report

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import wandb
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import data_loader
from src.ann.neural_network import NeuralNetwork


dataset = "mnist"  # Default dataset
X_train, Y_train, X_val, Y_val, X_test, Y_test = data_loader.load_dataset(dataset)


PROJECT_NAME = "mnist_mlp"

Data Exploration and Class Distribution

In [3]:
def _class_names(dataset_name):
    if dataset_name == "fashion_mnist":
        return [
            "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
            "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
        ]
    return [str(i) for i in range(10)]


def log_class_samples(X, y, dataset_name="mnist", samples_per_class=5):
    names = _class_names(dataset_name)
    y_labels = np.argmax(y, axis=1) if y.ndim == 2 else y

    table = wandb.Table(columns=["class_id", "class_name", "sample_idx", "image"])
    logged_counts = {i: 0 for i in range(10)}
    preview_rows = []

    for cls in range(10):
        idxs = np.where(y_labels == cls)[0][:samples_per_class]
        for sample_idx, idx in enumerate(idxs, start=1):
            img = X[idx].reshape(28, 28) if X.ndim == 2 else X[idx]
            table.add_data(cls, names[cls], sample_idx, wandb.Image(img, caption=f"{names[cls]} #{sample_idx}"))
            logged_counts[cls] += 1
            if len(preview_rows) < 12:
                preview_rows.append((cls, names[cls], sample_idx))

    table_key = f"{dataset_name}_class_samples"
    wandb.log({table_key: table})

    print(f"Logged table key: {table_key}")
    print(f"Total rows logged: {sum(logged_counts.values())}")
    print("Rows per class:", logged_counts)
    print("Preview of logged rows (class_id, class_name, sample_idx):")
    for row in preview_rows:
        print("  ", row)

    return table_key, logged_counts


try:
    if "X_train" not in globals() or "Y_train" not in globals():
        raise RuntimeError("Run the data-loading cell first so X_train and Y_train are available.")

    run = wandb.run if wandb.run is not None else wandb.init(project="mnist_mlp", name=f"data_exploration_{dataset}", reinit=True)
    print(f"W&B run: {run.name} ({run.id})")
    if run.url:
        print(f"Run URL: {run.url}")

    log_class_samples(X_train, Y_train, dataset_name=dataset, samples_per_class=5)
except Exception as e:
    print("W&B logging failed:", e)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


wandb: Currently logged in as: anandhakrishnanm21 (anandhakrishnanm21-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B run: data_exploration_mnist (k3tlw3iq)
Run URL: https://wandb.ai/anandhakrishnanm21-indian-institute-of-technology-madras/mnist_mlp/runs/k3tlw3iq
Logged table key: mnist_class_samples
Total rows logged: 50
Rows per class: {0: 5, 1: 5, 2: 5, 3: 5, 4: 5, 5: 5, 6: 5, 7: 5, 8: 5, 9: 5}
Preview of logged rows (class_id, class_name, sample_idx):
   (0, '0', 1)
   (0, '0', 2)
   (0, '0', 3)
   (0, '0', 4)
   (0, '0', 5)
   (1, '1', 1)
   (1, '1', 2)
   (1, '1', 3)
   (1, '1', 4)
   (1, '1', 5)
   (2, '2', 1)
   (2, '2', 2)


Hyperparameter Sweep

In [ ]:
from src.utils import data_loader
from src.ann.neural_network import NeuralNetwork


X_train, y_train, X_val, y_val, X_test, Y_test = data_loader.load_dataset(name = "mnist")


def train_with_wandb_for_sweep(exp_name = 'mnist_mlp_sweep'):
    #================KEY STEP====================
    wandb.init(project=PROJECT_NAME, name=exp_name)

    config = wandb.config
    
    model = NeuralNetwork(config)

    best_val_acc = 0

    for epoch in range(config.epochs):

        model.train(X_train, y_train, epochs=1, batch_size=config.batch_size, verbose=False)

        train_loss, train_acc = model.evaluate(X_train, y_train)
        val_loss, val_acc = model.evaluate(X_val, y_val)

        wandb.log({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model.save_model("best_model.npy")
    return best_val_acc

In [5]:
wandb.finish()

# Define sweep configuration
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {

        'learning_rate': {
            'values': [0.01, 0.05]  },

        'loss_type': {
            'values': ['mse',  'cross_entropy']  },

        'weight_init': {
            'values': ['random', 'xavier']  },

        'epochs': {'value': 20},

        'batch_size': {'values': [32, 64, 128]},

        #'optimizer': {'values': ['sgd', 'adam','nadam','rmsprop']},

        #'hidden_sizes': {'values': [[128, 64], [128, 128], [64, 64]]}
    }
}

#================KEY STEP===================
print("Creating sweep...")
sweep_id = wandb.sweep(
    sweep_config,
    project=PROJECT_NAME
)

# Calculate total runs
total_runs = 2 * 2 * 2 * 3
print(f"✓ Sweep created: {sweep_id}")
print(f"  Grid search will run {total_runs} experiments")
print("  Starting sweep...")

# Run sweep
wandb.agent(sweep_id, train_with_wandb_for_sweep, count=total_runs)

print("\n✓ Sweep completed!")
print("  Check your W&B dashboard!")

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Creating sweep...
Create sweep with ID: zkx9k4ld
Sweep URL: https://wandb.ai/anandhakrishnanm21-indian-institute-of-technology-madras/mnist_mlp/sweeps/zkx9k4ld
✓ Sweep created: zkx9k4ld
  Grid search will run 24 experiments
  Starting sweep...


wandb: Agent Starting Run: 888yf7ts with config:
wandb: 	batch_size: 32
wandb: 	epochs: 20
wandb: 	learning_rate: 0.01
wandb: 	loss_type: mse
wandb: 	weight_init: random
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


Traceback (most recent call last):
  File "d:\DL\DL_assignment_1\.dl1env\Lib\site-packages\wandb\agents\pyagent.py", line 304, in _run_job
    self._function()
  File "C:\Users\91701\AppData\Local\Temp\ipykernel_20640\343051862.py", line 20, in train_with_wandb_for_sweep
    model.train(X_train, y_train, epochs=1, batch_size=config.batch_size, verbose=False)
  File "D:\DL\DL_assignment_1\src\ann\neural_network.py", line 184, in train
    y_pred = self.forward(X_batch)
            ^^^^^^^^^^^^^^^^^^^^^
  File "D:\DL\DL_assignment_1\src\ann\neural_network.py", line 116, in forward
    z = layer.forward(a)
        ^^^^^^^^^^^^^^^^
  File "D:\DL\DL_assignment_1\src\ann\neural_layer.py", line 26, in forward
    return X @ self.W + self.b
           ~~^~~~~~~~
ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR Run 888yf7ts errored: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)
wandb: Agent Starting Run: 87dgoag3 with config:
wandb: 	batch_size: 32
wandb: 	epochs: 20
wandb: 	learning_rate: 0.01
wandb: 	loss_type: mse
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


...

Traceback (most recent call last):
  File "d:\DL\DL_assignment_1\.dl1env\Lib\site-packages\wandb\agents\pyagent.py", line 304, in _run_job
    self._function()
  File "C:\Users\91701\AppData\Local\Temp\ipykernel_20640\343051862.py", line 20, in train_with_wandb_for_sweep
    model.train(X_train, y_train, epochs=1, batch_size=config.batch_size, verbose=False)
  File "D:\DL\DL_assignment_1\src\ann\neural_network.py", line 184, in train
    y_pred = self.forward(X_batch)
            ^^^^^^^^^^^^^^^^^^^^^
  File "D:\DL\DL_assignment_1\src\ann\neural_network.py", line 116, in forward
    z = layer.forward(a)
        ^^^^^^^^^^^^^^^^
  File "D:\DL\DL_assignment_1\src\ann\neural_layer.py", line 26, in forward
    return X @ self.W + self.b
           ~~^~~~~~~~
ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: ERROR Run sn128yyf errored: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)
wandb: Agent Starting Run: efktpqn1 with config:
wandb: 	batch_size: 128
wandb: 	epochs: 20
wandb: 	learning_rate: 0.05
wandb: 	loss_type: cross_entropy
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\91701\_netrc.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▅▅▅▆▆▆▇▇▇▇▇██████
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇██████
val_loss,█▆▅▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch,19
train_acc,0.98824
train_loss,0.04648
val_acc,0.9735
val_loss,0.09814



✓ Sweep completed!
  Check your W&B dashboard!


The Optimizer Showdown

Vanishing Gradient Analysis

The ”Dead Neuron” Investigation

Loss Function Comparison

Global Performance Analysis

Error Analysis

Weight Initialization & Symmetry

The Fashion-MNIST Transfer Challenge